In [5]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple, Dict, Any

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [6]:
@dataclass
class NaoEofBaseConfig:
    # Domain and months
    lat_bounds: Tuple[float, float] = (20.0, 80.0)     # degrees north
    lon_bounds: Tuple[float, float] = (-90.0, 40.0)    # degrees east in [-180, 180] convention
    months_ndjfm: Tuple[int, ...] = (11, 12, 1, 2, 3)

    # Base period for climatology/anomalies
    base_start: str = "1985-01-01"
    base_end: str = "2014-12-31"

    # Names (override if needed)
    var_candidates: Tuple[str, ...] = ("PSL", "psl", "SLP", "slp", "msl", "MSL", "PMSL", "pmsl")
    time_name_candidates: Tuple[str, ...] = ("time", "Time", "date")
    lat_name_candidates: Tuple[str, ...] = ("lat", "latitude", "LAT", "nav_lat")
    lon_name_candidates: Tuple[str, ...] = ("lon", "longitude", "LON", "nav_lon")

    # Processing options
    detrend: bool = False             # typically False for NAO EOF base
    remove_global_mean: bool = False  # typically False for NAO EOF base
    enforce_unit_norm: bool = True    # normalize EOF1 so sum(eof^2)=1 in weighted space


class NaoEofBaseBuilder:
    """
    Build and save a NAO EOF base from monthly SLP/PSL.
    Leading EOF pattern over Atlantic sector (20–80N, 90W–40E), NDJFM only,
    anomalies relative to a base period (e.g., 1985–2014), area-weighted with sqrt(cos(lat)).
    """

    def __init__(self, config: Optional[NaoEofBaseConfig] = None):
        self.cfg = config or NaoEofBaseConfig()

    # -------------------------
    # Public API
    # -------------------------
    def build_and_save(
        self,
        in_file: str | Path,
        out_file: str | Path,
        var_name: Optional[str] = None,
        dataset_name: str = "UNKNOWN",
        extra_global_attrs: Optional[Dict[str, Any]] = None,
    ) -> xr.Dataset:
        """
        Build EOF base and write to NetCDF. Returns the saved Dataset.
        """
        in_file = Path(in_file)
        out_file = Path(out_file)
        out_file.parent.mkdir(parents=True, exist_ok=True)

        ds = xr.open_dataset(in_file)

        time_name = self._pick_name(ds, self.cfg.time_name_candidates, kind="time")
        lat_name = self._pick_name(ds, self.cfg.lat_name_candidates, kind="lat")
        lon_name = self._pick_name(ds, self.cfg.lon_name_candidates, kind="lon")

        da = self._select_var(ds, var_name)
        da = da.rename({time_name: "time", lat_name: "lat", lon_name: "lon"})

        da = self._subset_domain(da)
        da = self._ensure_lon_0360_or_180(da, target="180")  # we will subset with [-90, 40]
        da = self._subset_domain(da)  # re-apply after lon conversion if needed

        # NDJFM only
        da_ndjfm = da.sel(time=da["time"].dt.month.isin(self.cfg.months_ndjfm))

        # Base-period climatology for each calendar month (NDJFM months)
        base = da_ndjfm.sel(time=slice(self.cfg.base_start, self.cfg.base_end))
        if base.sizes.get("time", 0) < 5:
            raise ValueError(
                f"Too few base-period samples in {self.cfg.base_start}–{self.cfg.base_end}. "
                f"Check time coordinate and base period."
            )
        clim = base.groupby("time.month").mean("time")

        # Monthly anomalies
        anom = da_ndjfm.groupby("time.month") - clim

        # Optional: remove domain mean each time (usually not needed for NAO)
        if self.cfg.remove_global_mean:
            anom = anom - anom.mean(("lat", "lon"))

        # Optional: detrend (usually not needed; keep False unless you have a reason)
        if self.cfg.detrend:
            anom = self._detrend_linear(anom)

        # Build weighted anomaly matrix X (time, space)
        w_sqrt = np.sqrt(np.cos(np.deg2rad(anom["lat"]))).clip(min=0.0)
        # broadcast to lat-lon
        W = w_sqrt.broadcast_like(anom.isel(time=0))
        anom_w = anom * W

        # Mask any gridpoints that have any NaNs across time
        X = anom_w.stack(space=("lat", "lon"))
        valid = np.isfinite(X).all("time")
        Xv = X[:, valid]

        # Remove time-mean in weighted space (should be ~0 after anomaly, but keep safe)
        Xv = Xv - Xv.mean("time")

        # EOF via SVD: X = U S V^T  (time x space)
        # EOF1 in space = V[0, :]
        Xmat = Xv.data
        if hasattr(Xmat, "compute"):  # dask array
            Xmat = Xmat.compute()

        U, s, VT = np.linalg.svd(Xmat, full_matrices=False)
        pc1 = U[:, 0] * s[0]
        eof1_space = VT[0, :]

        # Explained variance fraction (covariance EOF)
        # total variance in weighted space:
        # Using singular values: variance explained by mode k proportional to s_k^2
        frac1 = (s[0] ** 2) / np.sum(s ** 2)

        # Put EOF back onto lat-lon grid (with NaNs where invalid)
        eof_full = xr.full_like(X.isel(time=0), np.nan).astype(np.float64)
        eof_full.loc[dict(space=valid)] = eof1_space
        eof_map = eof_full.unstack("space")

        # Normalize EOF to unit norm in weighted space if requested:
        # norm = sqrt(sum(eof^2)) in weighted grid space (already weighted anomalies were used,
        # but eof_map here is in weighted space too because derived from Xv).
        if self.cfg.enforce_unit_norm:
            norm = np.sqrt(np.nansum((eof_map.data) ** 2))
            if norm == 0 or not np.isfinite(norm):
                raise ValueError("EOF1 norm is invalid (0 or non-finite). Check data/masking.")
            eof_map = eof_map / norm
            pc1 = pc1 * norm  # keep X ≈ pc*eof consistent

        # Standardize sign (optional but helpful): make EOF have positive lobe near Azores-ish
        # This is arbitrary; we enforce a consistent sign using a reference point.
        # Choose (lat~35N, lon~-20E) within domain (Azores region). Flip if EOF there is negative.
        eof_ref = self._nearest_value(eof_map, lat=35.0, lon=-20.0)
        if np.isfinite(eof_ref) and eof_ref < 0:
            eof_map = -eof_map
            pc1 = -pc1

        # Pack output dataset
        out = xr.Dataset(
            data_vars=dict(
                nao_eof1=eof_map.rename("nao_eof1"),
                nao_pc1=("time", pc1.astype(np.float64)),
                explained_variance_fraction=xr.DataArray(frac1.astype(np.float64)),
                base_climatology=clim.rename("base_climatology"),
                lat_weights_sqrtcos=w_sqrt.rename("lat_weights_sqrtcos"),
            ),
            coords=dict(
                time=anom["time"],
                lat=anom["lat"],
                lon=anom["lon"],
                month=clim["month"],
            ),
            attrs=self._make_attrs(
                dataset_name=dataset_name,
                in_file=str(in_file),
                var_used=str(da.name),
                extra=extra_global_attrs or {},
            ),
        )

        # Make variables CF-ish
        out["nao_eof1"].attrs.update(
            dict(
                long_name="NAO leading EOF pattern (weighted space, unit norm)" if self.cfg.enforce_unit_norm
                else "NAO leading EOF pattern (weighted space)",
                description="EOF1 from NDJFM monthly SLP anomalies over Atlantic sector; "
                            "anomalies computed relative to base_climatology; "
                            "area-weighted with sqrt(cos(lat)).",
                units="1",
            )
        )
        out["nao_pc1"].attrs.update(dict(long_name="NAO PC1 time series (projection in weighted space)", units="Pa"))
        out["explained_variance_fraction"].attrs.update(dict(long_name="EOF1 explained variance fraction", units="1"))
        out["base_climatology"].attrs.update(dict(long_name="Base-period monthly climatology (NDJFM months only)"))
        out["lat_weights_sqrtcos"].attrs.update(dict(long_name="Latitude weights sqrt(cos(lat))", units="1"))

        # Save
        encoding = {
            "nao_eof1": {"zlib": True, "complevel": 4},
            "nao_pc1": {"zlib": True, "complevel": 4},
            "base_climatology": {"zlib": True, "complevel": 4},
        }
        out.to_netcdf(out_file, encoding=encoding)

        ds.close()
        return out

    # -------------------------
    # Helpers
    # -------------------------
    def _pick_name(self, ds: xr.Dataset, candidates: Tuple[str, ...], kind: str) -> str:
        for name in candidates:
            if name in ds.coords or name in ds.dims:
                return name
        # If not found in coords/dims, check variables (some files store lat/lon as data_vars)
        for name in candidates:
            if name in ds.variables:
                return name
        raise KeyError(f"Could not find {kind} name among candidates: {candidates}. "
                       f"Available: dims={list(ds.dims)}, coords={list(ds.coords)}, vars={list(ds.data_vars)}")

    def _select_var(self, ds: xr.Dataset, var_name: Optional[str]) -> xr.DataArray:
        if var_name is not None:
            if var_name not in ds:
                raise KeyError(f"Requested var_name='{var_name}' not found. Available: {list(ds.data_vars)}")
            return ds[var_name]

        for v in self.cfg.var_candidates:
            if v in ds.data_vars:
                return ds[v]
        raise KeyError(f"Could not find SLP variable among {self.cfg.var_candidates}. "
                       f"Available data_vars: {list(ds.data_vars)}")

    def _ensure_lon_0360_or_180(self, da: xr.DataArray, target: str = "180") -> xr.DataArray:
        lon = da["lon"].values
        if lon.ndim != 1:
            return da

        lon_min = np.nanmin(lon)
        lon_max = np.nanmax(lon)

        if target == "180":
            # Convert to [-180,180)
            if lon_max > 180.0 + 1e-6:
                lon_new = ((lon + 180.0) % 360.0) - 180.0
                da = da.assign_coords(lon=lon_new).sortby("lon")
        elif target == "360":
            # Convert to [0,360)
            if lon_min < 0.0 - 1e-6:
                lon_new = lon % 360.0
                da = da.assign_coords(lon=lon_new).sortby("lon")
        else:
            raise ValueError("target must be '180' or '360'")
        return da

    def _subset_domain(self, da: xr.DataArray) -> xr.DataArray:
        lat0, lat1 = self.cfg.lat_bounds
        lon0, lon1 = self.cfg.lon_bounds

        # Make sure slice works regardless of lat ordering
        if da["lat"][0] < da["lat"][-1]:
            da = da.sel(lat=slice(lat0, lat1))
        else:
            da = da.sel(lat=slice(lat1, lat0))

        # lon assumed in [-180,180] if target="180"
        # Handle potential wrap if lon0 > lon1 (not the case here, but keep robust)
        if lon0 <= lon1:
            da = da.sel(lon=slice(lon0, lon1))
        else:
            da1 = da.sel(lon=slice(lon0, 180.0))
            da2 = da.sel(lon=slice(-180.0, lon1))
            da = xr.concat([da1, da2], dim="lon").sortby("lon")

        return da

    def _detrend_linear(self, da: xr.DataArray) -> xr.DataArray:
        # simple linear detrend along time, preserving dims
        t = xr.DataArray(np.arange(da.sizes["time"]), dims=("time",), coords={"time": da["time"]})
        # polyfit across time for each gridpoint
        p = da.polyfit(dim="time", deg=1)
        trend = xr.polyval(t, p["polyfit_coefficients"])
        return da - trend

    def _nearest_value(self, da2d: xr.DataArray, lat: float, lon: float) -> float:
        # lon may be in [-180,180]; just pick nearest
        p = da2d.sel(lat=lat, lon=lon, method="nearest")
        v = float(p.values)
        return v

    def _make_attrs(
        self, dataset_name: str, in_file: str, var_used: str, extra: Dict[str, Any]
    ) -> Dict[str, Any]:
        attrs: Dict[str, Any] = dict(
            nao_definition="EOF1 of monthly NDJFM SLP anomalies over 20–80N, 90W–40E; index via projection onto EOF1",
            dataset_name=dataset_name,
            input_file=in_file,
            variable_used=var_used,
            base_period_start=self.cfg.base_start,
            base_period_end=self.cfg.base_end,
            months_ndjfm=",".join(map(str, self.cfg.months_ndjfm)),
            lat_bounds=str(self.cfg.lat_bounds),
            lon_bounds=str(self.cfg.lon_bounds),
            weighting="sqrt(cos(lat))",
            eof_method="SVD on weighted anomalies (covariance EOF)",
            normalized_unit_norm=str(self.cfg.enforce_unit_norm),
            detrend=str(self.cfg.detrend),
            remove_domain_mean=str(self.cfg.remove_global_mean),
        )
        attrs.update(extra)
        return attrs


In [7]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "nao"
    out_dir.mkdir(parents=True, exist_ok=True)

    # training period (overlap)
    train_start = "1985-01-01"
    train_end   = "2014-12-31"
    period = "1985-2014"

    # input (NOAA-20C monthly PSL)
    in_file = "/lcrc/soft/climate/e3sm_diags_data/obs_for_e3sm_diags/time-series/NOAA-20C/psl_183601_201512.nc"

    # output NAO EOF base
    out_nc = out_dir / f"nao_eof_base_NOAA20C_{period}_NDJFM_20-80N_90W-40E.nc"

    cfg = NaoEofBaseConfig(
        base_start=train_start,
        base_end=train_end,
        # domain/months are already set to (20–80N, 90W–40E) and NDJFM by default,
        # but you can override here if you want.
        detrend=False,
        remove_global_mean=False,
        enforce_unit_norm=True,
    )

    builder = NaoEofBaseBuilder(cfg)

    out = builder.build_and_save(
        in_file=in_file,
        out_file=out_nc,
        var_name="psl",          # if your file uses another name, change this or set to None for auto-detect
        dataset_name="NOAA-20C",
        extra_global_attrs={
            "note": "NAO EOF base built from NOAA-20C PSL; NDJFM only; anomalies computed from 1985–2014 monthly climatology."
        },
    )

    print(f"Wrote: {out_nc}")
    print(out)

Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/nao/nao_eof_base_NOAA20C_1985-2014_NDJFM_20-80N_90W-40E.nc
<xarray.Dataset> Size: 83kB
Dimensions:                      (lat: 60, lon: 40, time: 900, month: 5)
Coordinates:
  * lat                          (lat) float64 480B 20.5 21.5 22.5 ... 78.5 79.5
  * lon                          (lon) float64 320B 0.5 1.5 2.5 ... 38.5 39.5
  * time                         (time) datetime64[ns] 7kB 1836-01-01 ... 201...
  * month                        (month) int64 40B 1 2 3 11 12
Data variables:
    nao_eof1                     (lat, lon) float64 19kB 0.006757 ... -0.01579
    nao_pc1                      (time) float64 7kB 6.101e+03 ... 2.056e+04
    explained_variance_fraction  float64 8B 0.4783
    base_climatology             (month, lat, lon) float32 48kB 1.016e+05 ......
    lat_weights_sqrtcos          (lat) float64 480B 0.9678 0.9646 ... 0.4269
Attributes: (12/15)
    nao_definition:        EOF1 of mon